# Task 06 - Training and Optimisation
Change one hyperparameter at a time and report only completed trials.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running locally
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import pandas as pd
from src.config import RAW_DATA_DIR
from src.data import load_ogbn_arxiv
from src.models import GCN
from src.training import fit, set_seed
from src.training.hyperparameter_tuning import parameter_grid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset, data, split_idx = load_ogbn_arxiv(RAW_DATA_DIR)
data = data.to(device)
split_idx = {k: v.to(device) for k, v in split_idx.items()}

trials = parameter_grid({'hidden_channels': [128, 256], 'dropout': [0.3, 0.5], 'learning_rate': [0.01]})
results = []
for params in trials:
    set_seed(42)
    model = GCN(data.num_features, params['hidden_channels'], dataset.num_classes, params['dropout']).to(device)
    history = fit(model, data, split_idx, epochs=20, learning_rate=params['learning_rate'])
    results.append({**params, 'validation_accuracy': history['validation_accuracy'].max()})

tuning = pd.DataFrame(results).sort_values('validation_accuracy', ascending=False)
results_dir = PROJECT_ROOT / 'results' / 'training'
results_dir.mkdir(parents=True, exist_ok=True)
tuning.to_csv(results_dir / 'hyperparameter_trials.csv', index=False)
tuning

C:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\ogb\nodeproppred\dataset_pyg.py:91: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  train_idx = torch.from_numpy(pd.read_csv(osp.join(path, 'train.csv.gz'), compression='gzip', header = None).values.T[0]).to(torch.long)
